# Feature Engineering — previous_application.csv

**¿Qué es esta tabla?**
Contiene todas las solicitudes de crédito anteriores que cada cliente hizo en Home Credit
(no en el buró externo — esas estaban en bureau.csv). Cada fila es una solicitud anterior.

**¿Por qué importa?**
El comportamiento pasado con la misma institución es uno de los predictores más fuertes
de comportamiento futuro. Si un cliente ya pidió créditos aquí antes y los pagó bien,
eso es una señal muy diferente a si los rechazaron o los canceló.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)
DATA_PATH = '../data/raw/'

print('Librerías cargadas.')

Librerías cargadas.


## 1. Exploración

In [2]:
prev = pd.read_csv(DATA_PATH + 'previous_application.csv')
print(f'Shape: {prev.shape}')
print(f'Clientes únicos: {prev["SK_ID_CURR"].nunique():,}')
print(f'Solicitudes por cliente (promedio): {len(prev) / prev["SK_ID_CURR"].nunique():.1f}')
print(f'\nNAME_CONTRACT_STATUS únicos: {prev["NAME_CONTRACT_STATUS"].value_counts().to_dict()}')
print(f'\nMissings >10%:')
miss = prev.isnull().mean()
print(miss[miss > 0.1].round(3).sort_values(ascending=False))

Shape: (1670214, 37)
Clientes únicos: 338,857
Solicitudes por cliente (promedio): 4.9

NAME_CONTRACT_STATUS únicos: {'Approved': 1036781, 'Canceled': 316319, 'Refused': 290678, 'Unused offer': 26436}

Missings >10%:
RATE_INTEREST_PRIMARY        0.996
RATE_INTEREST_PRIVILEGED     0.996
AMT_DOWN_PAYMENT             0.536
RATE_DOWN_PAYMENT            0.536
NAME_TYPE_SUITE              0.491
DAYS_FIRST_DRAWING           0.403
DAYS_FIRST_DUE               0.403
DAYS_LAST_DUE_1ST_VERSION    0.403
DAYS_LAST_DUE                0.403
DAYS_TERMINATION             0.403
NFLAG_INSURED_ON_APPROVAL    0.403
AMT_GOODS_PRICE              0.231
AMT_ANNUITY                  0.223
CNT_PAYMENT                  0.223
dtype: float64


## 2. Feature Engineering

In [3]:
# Reemplazar placeholder 365243 en columnas de días
days_cols = ['DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE', 'DAYS_LAST_DUE_1ST_VERSION',
             'DAYS_LAST_DUE', 'DAYS_TERMINATION']
for col in days_cols:
    prev[col] = prev[col].replace(365243, np.nan)

# Feature derivada: ratio entre crédito aprobado y crédito solicitado
# Si aprobaron menos de lo que pidió = señal de que el analista vio riesgo
prev['PREV_CREDIT_TO_APP_RATIO'] = prev['AMT_CREDIT'] / (prev['AMT_APPLICATION'] + 1)

# Feature derivada: diferencia entre último pago programado y primer pago
prev['PREV_DAYS_DURATION'] = prev['DAYS_LAST_DUE'] - prev['DAYS_FIRST_DUE']

prev_agg = prev.groupby('SK_ID_CURR').agg(
    # Cantidad y resultado de solicitudes
    PREV_COUNT=('SK_ID_PREV', 'count'),
    PREV_APPROVED_COUNT=('NAME_CONTRACT_STATUS', lambda x: (x == 'Approved').sum()),
    PREV_REFUSED_COUNT=('NAME_CONTRACT_STATUS', lambda x: (x == 'Refused').sum()),
    PREV_CANCELED_COUNT=('NAME_CONTRACT_STATUS', lambda x: (x == 'Canceled').sum()),

    # Montos
    PREV_AMT_CREDIT_MEAN=('AMT_CREDIT', 'mean'),
    PREV_AMT_CREDIT_MAX=('AMT_CREDIT', 'max'),
    PREV_AMT_APPLICATION_MEAN=('AMT_APPLICATION', 'mean'),
    PREV_AMT_DOWN_PAYMENT_MEAN=('AMT_DOWN_PAYMENT', 'mean'),
    PREV_AMT_ANNUITY_MEAN=('AMT_ANNUITY', 'mean'),

    # Ratio aprobado/solicitado
    PREV_CREDIT_TO_APP_RATIO_MEAN=('PREV_CREDIT_TO_APP_RATIO', 'mean'),
    PREV_CREDIT_TO_APP_RATIO_MIN=('PREV_CREDIT_TO_APP_RATIO', 'min'),

    # Días
    PREV_DAYS_DECISION_MEAN=('DAYS_DECISION', 'mean'),
    PREV_DAYS_DURATION_MEAN=('PREV_DAYS_DURATION', 'mean'),

    # Tasa de interés
    PREV_RATE_DOWN_PAYMENT_MEAN=('RATE_DOWN_PAYMENT', 'mean'),
).reset_index()

# Ratio de rechazo — cuántas de sus solicitudes anteriores fueron rechazadas
prev_agg['PREV_REFUSED_RATIO'] = prev_agg['PREV_REFUSED_COUNT'] / (prev_agg['PREV_COUNT'] + 1)
prev_agg['PREV_APPROVED_RATIO'] = prev_agg['PREV_APPROVED_COUNT'] / (prev_agg['PREV_COUNT'] + 1)

print(f'Shape prev_agg: {prev_agg.shape}')
print(f'Features generadas: {prev_agg.shape[1] - 1}')
prev_agg.head(3)

Shape prev_agg: (338857, 17)
Features generadas: 16


,SK_ID_CURR,PREV_COUNT,PREV_APPROVED_COUNT,PREV_REFUSED_COUNT,PREV_CANCELED_COUNT,PREV_AMT_CREDIT_MEAN,PREV_AMT_CREDIT_MAX,PREV_AMT_APPLICATION_MEAN,PREV_AMT_DOWN_PAYMENT_MEAN,PREV_AMT_ANNUITY_MEAN,PREV_CREDIT_TO_APP_RATIO_MEAN,PREV_CREDIT_TO_APP_RATIO_MIN,PREV_DAYS_DECISION_MEAN,PREV_DAYS_DURATION_MEAN,PREV_RATE_DOWN_PAYMENT_MEAN,PREV_REFUSED_RATIO,PREV_APPROVED_RATIO
0,100001,1,1,0,0,23787.0,23787.0,24835.5,2520.0,3951.000,0.957744,0.957744,-1740.0,90.0,0.104326,0.0,0.50
1,100002,1,1,0,0,179055.0,179055.0,179055.0,0.0,9251.775,0.999994,0.999994,-606.0,540.0,0.000000,0.0,0.50
2,100003,3,3,0,0,484191.0,1035882.0,435436.5,3442.5,56553.990,1.057658,0.988999,-1305.0,220.0,0.050030,0.0,0.75


## 3. Guardar y validar

In [4]:
prev_agg.to_parquet('../data/processed/prev_app_features.parquet', index=False)
print('Guardado en data/processed/prev_app_features.parquet')

# Validar correlación con TARGET
app = pd.read_csv(DATA_PATH + 'application_train.csv', usecols=['SK_ID_CURR', 'TARGET'])
val = app.merge(prev_agg, on='SK_ID_CURR', how='left')

print(f'\nClientes SIN historial en prev_app: {val["PREV_COUNT"].isna().sum():,} ({val["PREV_COUNT"].isna().mean()*100:.1f}%)')

print('\nCorrelación con TARGET (top features):')
corr = val.drop(columns='SK_ID_CURR').corr()['TARGET'].drop('TARGET')
print(corr.abs().sort_values(ascending=False).head(10).round(4))

Guardado en data/processed/prev_app_features.parquet

Clientes SIN historial en prev_app: 16,454 (5.4%)

Correlación con TARGET (top features):
PREV_REFUSED_RATIO             0.0785
PREV_APPROVED_RATIO            0.0775
PREV_REFUSED_COUNT             0.0645
PREV_DAYS_DECISION_MEAN        0.0469
PREV_AMT_ANNUITY_MEAN          0.0349
PREV_RATE_DOWN_PAYMENT_MEAN    0.0336
PREV_APPROVED_COUNT            0.0316
PREV_AMT_DOWN_PAYMENT_MEAN     0.0246
PREV_AMT_APPLICATION_MEAN      0.0218
PREV_COUNT                     0.0198
Name: TARGET, dtype: float64
